# M5: исследовательский анализ розничных продаж

Цель EDA анализа: понять структуру спроса,
проверить предпосылки прогнозирования и сохранить полезные результаты.
Единица наблюдения: продажи одного товара в одном магазине за день; целевая переменная: количество единиц

Исследуем весь `sales_train_validation.csv`: 30 490 рядов
Основной EDA заканчивается 31.01.2016. Следующие три непересекающихся окна по 28 дней
оставлены для проверки простых прогнозных гипотез. Гиперпараметры по ним не подбираются.

Вопросы, сформулированный для исследования:
1. Какие нули связаны с отсутствием наблюдаемой цены, а какие остаются после появления товара?
2. Насколько итоговый ряд скрывает неоднородность товаров и магазинов?
3. Что дают недельная сезонность, SNAP, праздники и изменения цены?
4. Переносятся ли наблюдения EDA на будущие периоды, какие метрики и проверки нужны в эксплуатации?

## 1. Настройки

Загрузка продаж частями: полное преобразование. Seed фиксирует bootstrap и выбор примеров

In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from scipy import stats
from statsmodels.tsa.seasonal import STL
import statsmodels.formula.api as smf

STARTED = time.perf_counter()
SEED, HORIZON, N_FOLDS = 42, 28, 3
rng = np.random.default_rng(SEED)
cwd = Path.cwd().resolve()
roots = [p for base in [cwd, *cwd.parents] for p in [base, base / 'repository']]
ROOT = next((p for p in roots if (p / 'docker-compose.yaml').is_file()
             and (p / 'notebooks' / 'M5_EDA_MLOps.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте notebook из клонированного repository или его notebooks/.')
data_candidates = ([Path(os.environ['M5_DATA_DIR']).expanduser().resolve()]
                   if os.environ.get('M5_DATA_DIR') else [ROOT / 'data', ROOT.parent / 'data'])
required_csv = ['calendar.csv', 'sales_train_validation.csv', 'sell_prices.csv']
DATA = next((p for p in data_candidates if (p / 'manifest.json').is_file()
             and all((p / 'raw' / name).is_file() for name in required_csv)), None)
if DATA is None:
    raise FileNotFoundError('Данные не найдены. Из repository: python scripts/download_m5.py')
RAW = DATA / 'raw'
OUT = ROOT / 'artifacts' / 'eda'
FIGURES, TABLES = OUT / 'figures', OUT / 'tables'
for folder in [FIGURES, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='colorblind', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (11, 4), 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 12})
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def table(frame, name, show=True):
    frame.to_csv(TABLES / f'{name}.csv', index=False, encoding='utf-8-sig')
    if show:
        display(frame)
    return frame

def note(text):
    display(Markdown(text))

def file_hash(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

packages = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy', 'statsmodels']
versions = {}
for name in packages:
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        versions[name] = None
environment = {'python': sys.version, 'executable': sys.executable,
               'platform': platform.platform(), 'seed': SEED, 'packages': versions}
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(f'Python {platform.python_version()}; seed={SEED}')
print(f'Interpreter: {sys.executable}')
print(f'Data: {DATA}')
print(f'Results: {OUT}')

## 2. Загрузка, схема и контракт качества

`sales` связывается с календарём по `d`, а с ценами по `(store_id, item_id, wm_yr_wk)`.
Цена недельная, продажи дневные. Пустой `event_name` означает отсутствие события;
пустую цену нельзя автоматически заполнять нулём. Проверяем ключи, диапазоны и связи
до статистических выводов. Проверки структуры охватывают все файлы

In [ ]:
manifest = json.loads((DATA / 'manifest.json').read_text(encoding='utf-8'))
hash_checks = []
for record in manifest['files']:
    path = RAW / record['filename']
    actual = file_hash(path)
    assert actual == record['sha256'], f'Изменён исходный файл: {path.name}'
    hash_checks.append({'file': path.name, 'MB': path.stat().st_size / 1e6, 'sha256': actual})
table(pd.DataFrame(hash_checks), 'source_checksums')
(OUT / 'data_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

calendar = pd.read_csv(RAW / 'calendar.csv', parse_dates=['date'])
header = pd.read_csv(RAW / 'sales_train_validation.csv', nrows=0).columns.tolist()
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in header if c.startswith('d_')]
assert day_cols == [f'd_{i}' for i in range(1, 1914)]
Y = np.empty((30490, len(day_cols)), dtype=np.int16)
metadata_parts, position = [], 0
for chunk in pd.read_csv(RAW / 'sales_train_validation.csv', chunksize=2048,
                         dtype={c: 'int32' for c in day_cols}):
    block = chunk[day_cols].to_numpy()
    assert np.isfinite(block).all() and block.min() >= 0 and block.max() <= np.iinfo(np.int16).max
    Y[position:position + len(chunk)] = block
    metadata_parts.append(chunk[id_cols].copy())
    position += len(chunk)
assert position == len(Y)
meta = pd.concat(metadata_parts, ignore_index=True)
del chunk, block, metadata_parts
for c in id_cols[1:]:
    meta[c] = meta[c].astype('category')
price_dtypes = {'store_id': pd.CategoricalDtype(meta.store_id.cat.categories),
                'item_id': pd.CategoricalDtype(meta.item_id.cat.categories),
                'wm_yr_wk': 'int32', 'sell_price': 'float32'}
prices = pd.read_csv(RAW / 'sell_prices.csv', dtype=price_dtypes)
cal = calendar.set_index('d').loc[day_cols].reset_index()
checks = {
    '30490 рядов и 1913 дней': Y.shape == (30490, 1913),
    'id уникален': meta.id.is_unique,
    'товар × магазин уникален': not meta.duplicated(['item_id', 'store_id']).any(),
    'ключ календаря уникален': calendar.d.is_unique and calendar.date.is_unique,
    'календарь ежедневный без разрывов': calendar.date.diff().dropna().eq(pd.Timedelta(days=1)).all(),
    'ключ недельной цены уникален': not prices.duplicated(['store_id', 'item_id', 'wm_yr_wk']).any(),
    'в таблице цен нет пропусков': not prices.isna().any().any(),
    'все заданные цены положительны': prices.sell_price.gt(0).all(),
    'недели цен есть в календаре': prices.wm_yr_wk.isin(calendar.wm_yr_wk).all(),
    'SNAP бинарный': calendar[['snap_CA', 'snap_TX', 'snap_WI']].isin([0, 1]).all().all(),
    'иерархия item → department → category': meta.groupby('item_id', observed=True).dept_id.nunique().eq(1).all()
        and meta.groupby('dept_id', observed=True).cat_id.nunique().eq(1).all(),
    'иерархия store → state': meta.groupby('store_id', observed=True).state_id.nunique().eq(1).all(),
}
quality = table(pd.DataFrame({'check': list(checks), 'passed': list(checks.values())}), 'data_quality')
assert quality.passed.all(), 'Нарушен контракт данных: см. таблицу выше.'
dimensions = table(pd.DataFrame([
    {'entity': 'sales', 'rows': len(meta), 'columns': len(header)},
    {'entity': 'calendar', 'rows': len(calendar), 'columns': len(calendar.columns)},
    {'entity': 'prices', 'rows': len(prices), 'columns': len(prices.columns)},
]), 'dimensions')
note(f'**Проверки пройдены.** {meta.item_id.nunique():,} товаров, {meta.store_id.nunique()} магазинов, '
     f'{meta.state_id.nunique()} штата, {meta.dept_id.nunique()} отделов. Матрица продаж занимает '
     f'{Y.nbytes / 1024**2:.1f} MiB; полная длинная таблица не создаётся.')

## 3. Временной протокол

Последние 84 дня не используются для выбора примеров, сегментов, объясняющих моделей или порогов EDA.
В rolling-origin проверке прогноз строится сразу на 28 дней; на следующей точке старта уже наблюдённый
период можно включить в историю. Это имитация ежемесячного пересчёта, без перемешивания дат

In [ ]:
T = Y.shape[1]
CUT = T - N_FOLDS * HORIZON
train = Y[:, :CUT]
eda_cal = cal.iloc[:CUT].copy()
dates = pd.DatetimeIndex(eda_cal.date)
splits = pd.DataFrame([
    {'part': 'EDA', 'start': cal.date.iloc[0], 'end': cal.date.iloc[CUT - 1], 'days': CUT},
    *[{'part': f'backtest_{i + 1}', 'start': cal.date.iloc[CUT + i * HORIZON],
       'end': cal.date.iloc[CUT + (i + 1) * HORIZON - 1], 'days': HORIZON} for i in range(N_FOLDS)]
])
table(splits, 'temporal_split')
assert eda_cal.date.max() < splits.loc[1, 'start']

weeks = pd.Index(cal.wm_yr_wk.drop_duplicates())
keys = pd.MultiIndex.from_frame(meta[['store_id', 'item_id']])
price_wide = prices.pivot(index=['store_id', 'item_id'], columns='wm_yr_wk', values='sell_price')
P = price_wide.reindex(index=keys, columns=weeks).to_numpy(dtype=np.float32)
week_index = weeks.get_indexer(cal.wm_yr_wk)
del price_wide, prices
gc.collect()
assert P.shape == (len(meta), len(weeks))

## 4. Масштаб, неоднородность и концентрация

Для начала проведём сравнение магазинов и категорий, затем концентрацию продаж на уровне товар x магазин.
Все доли здесь измеряются в шт, а не выручке. Для проверки концентрации используем
последние 365 дней EDA, чтобы история появления товара меньше влияла на ранжирование

In [ ]:
total = train.sum(axis=0, dtype=np.int64)
by_cat = {str(k): train[ix].sum(axis=0, dtype=np.int64)
          for k, ix in meta.groupby('cat_id', observed=True).indices.items()}
by_store = {str(k): train[ix].sum(axis=0, dtype=np.int64)
            for k, ix in meta.groupby('store_id', observed=True).indices.items()}
recent_units = train[:, -365:].sum(axis=1, dtype=np.int64)
mix = meta[['cat_id', 'store_id']].copy()
mix['units_last_365'] = recent_units
mix_table = mix.groupby(['store_id', 'cat_id'], observed=True).units_last_365.sum().unstack()
table(mix_table.reset_index(), 'store_category_units')
ranked = np.sort(recent_units)[::-1]
concentration = np.cumsum(ranked) / ranked.sum()
top20_share = float(concentration[int(np.ceil(len(ranked) * .2)) - 1])
n80 = int(np.searchsorted(concentration, .8) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for cat, values in by_cat.items():
    axes[0].plot(dates, pd.Series(values).rolling(28).mean(), label=cat)
axes[0].set(title='Продажи по категориям: среднее за 28 дней', ylabel='Единиц в день')
axes[0].legend(fontsize=8)
(mix_table.div(mix_table.sum(axis=1), axis=0) * 100).plot.bar(stacked=True, ax=axes[1], width=.8)
axes[1].set(title='Состав продаж магазинов: последний год', ylabel='% единиц', xlabel='')
axes[1].legend(fontsize=7)
axes[2].plot(np.arange(1, len(ranked) + 1) / len(ranked), concentration)
axes[2].plot([0, 1], [0, 1], '--', color='gray')
axes[2].axvline(.2, linestyle=':', color='gray')
axes[2].set(title='Концентрация товар × магазин', xlabel='Доля рядов, от крупнейших', ylabel='Накопленная доля продаж')
savefig(fig, '01_scale_and_concentration')
table(pd.DataFrame([{'top20_units_share': top20_share, 'series_for_80pct': n80,
                     'series_for_80pct_share': n80 / len(meta)}]), 'concentration')
note(f'Вывод: 20% рядов дают {top20_share:.1%} продаж последнего года; для 80% объёма достаточно '
     f'{n80:,} рядов ({n80 / len(meta):.1%}). Средняя ошибка по рядам и ошибка по суммарному объёму '
     'отвечают на разные вопросы, поэтому далее используем обе.')

## 5. Нулевые продажи: спрос или отсутствие наблюдаемого ассортимента?

Гипотеза: доля нулей завышает прерывистость спроса, если включать дни до появления товара.
Сравним три определения: все дни, дни после первой положительной продажи и дни с известной ценой.
Цена в M5 является косвенным признаком представленности товара. Её наличие не гарантирует наличие остатков;
по этим данным нельзя доказать stockout или отделить потерянный спрос от "истинного" нуля

In [ ]:
N = len(meta)
first_sale = np.full(N, CUT, dtype=np.int32)
positive_days = np.zeros(N, dtype=np.int32)
sum_positive_sq = np.zeros(N, dtype=np.float64)
for lo in range(0, N, 1024):
    block = train[lo:lo + 1024]
    pos = block > 0
    counts = pos.sum(axis=1)
    first_sale[lo:lo + len(block)] = np.where(counts > 0, pos.argmax(axis=1), CUT)
    positive_days[lo:lo + len(block)] = counts
    sum_positive_sq[lo:lo + len(block)] = np.square(block.astype(np.float32)).sum(axis=1, dtype=np.float64)
del block, pos
age = CUT - first_sale
known_days = np.zeros(N, dtype=np.int32)
known_zero_days = np.zeros(N, dtype=np.int32)
first_price_day = np.full(N, CUT, dtype=np.int32)
no_price_sales = 0
active_count = np.empty(CUT, dtype=np.int32)
weekly_sales, eda_week_ids, week_day_counts = [], [], []
for w in np.unique(week_index[:CUT]):
    indices = np.flatnonzero(week_index[:CUT] == w)
    block = train[:, indices]
    known = np.isfinite(P[:, w])
    known_days += known * len(indices)
    known_zero_days += (block == 0).sum(axis=1).astype(np.int32) * known
    first_price_day = np.where(known & (first_price_day == CUT), indices[0], first_price_day)
    no_price_sales += int(block[~known].sum(dtype=np.int64))
    active_count[indices] = known.sum()
    weekly_sales.append(block.sum(axis=1, dtype=np.int32))
    eda_week_ids.append(w)
    week_day_counts.append(len(indices))
W = np.stack(weekly_sales, axis=1)
eda_week_ids = np.array(eda_week_ids)
week_day_counts = np.array(week_day_counts)
del weekly_sales, block
all_zero_share = float(1 - positive_days.sum() / train.size)
post_sale_zero_share = float(1 - positive_days.sum() / age.sum())
known_zero_share = float(known_zero_days.sum() / known_days.sum())
missing_price_share = float(1 - known_days.sum() / train.size)
zero_table = table(pd.DataFrame([
    {'population': 'Все товаро-дни', 'exposure_days': train.size, 'zero_share': all_zero_share},
    {'population': 'После первой продажи', 'exposure_days': age.sum(), 'zero_share': post_sale_zero_share},
    {'population': 'Цена известна', 'exposure_days': known_days.sum(), 'zero_share': known_zero_share},
]), 'zero_sales_exposure')
assert no_price_sales == 0, 'Есть ненулевые продажи без цены — требуется расследование.'
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=zero_table, x='population', y='zero_share', ax=axes[0])
axes[0].set(title='Доля нулей зависит от определения истории', xlabel='', ylabel='Доля нулевых товаро-дней', ylim=(0, 1))
axes[0].tick_params(axis='x', labelsize=9)
axes[1].plot(dates, active_count)
axes[1].set(title='Ряды с наблюдаемой ценой', ylabel='Количество товар × магазин')
savefig(fig, '02_zeros_and_assortment')
note(f'Результат эксперимента: Нули: {all_zero_share:.1%} на всей истории и {known_zero_share:.1%} '
     f'при известной цене. Цена отсутствует у {missing_price_share:.1%} товаро-дней; '
     f'продаж без цены: {no_price_sales:,}. Это основание различать экспозицию и спрос, '
     'но не основание удалять все нулевые продажи.')

## 6. Прерывистость и типы рядов

Сегментируем историю после первой продажи, где ADI, это число наблюдаемых дней на один день с продажами;
CV^2, это квадрат коэффициента вариации ненулевых продаж. Границы 1.32 и 0.49 как эвристика (основываясь на статьях от 2000ого года),
а не обученные оптимальные пороги. Ряды с историей короче 90 дней или менее 10 дней продаж
выделяем отдельно

In [ ]:
units = train.sum(axis=1, dtype=np.int64)
mean_positive = np.divide(units, positive_days, out=np.full(N, np.nan), where=positive_days > 0)
var_positive = np.maximum(np.divide(sum_positive_sq, positive_days,
                                   out=np.full(N, np.nan), where=positive_days > 0) - mean_positive**2, 0)
adi = np.divide(age, positive_days, out=np.full(N, np.nan), where=positive_days > 0)
cv2 = var_positive / mean_positive**2
segment = np.select([
    (age < 90) | (positive_days < 10),
    (adi < 1.32) & (cv2 < .49),
    (adi < 1.32) & (cv2 >= .49),
    (adi >= 1.32) & (cv2 < .49),
], ['short_history', 'smooth', 'erratic', 'intermittent'], default='lumpy')
profile = meta.copy()
profile['first_sale_day'] = first_sale + 1
profile['age_days'] = age
profile['ADI'], profile['CV2'] = adi, cv2
profile['segment'] = segment
profile['units_last_365'] = recent_units
profile['zero_share_known_price'] = np.divide(known_zero_days, known_days,
                                             out=np.full(N, np.nan), where=known_days > 0)
table(profile, 'series_profile', show=False)
segments = profile.groupby('segment').agg(series=('id', 'size'), units=('units_last_365', 'sum')).reset_index()
segments['series_share'] = segments.series / N
segments['units_share'] = segments.units / recent_units.sum()
table(segments, 'demand_segments')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sample = profile.sample(min(6000, N), random_state=SEED)
sns.scatterplot(data=sample, x='ADI', y='CV2', hue='segment', s=9, alpha=.4, ax=axes[0])
axes[0].set(xscale='log', yscale='log', title='ADI–CV²: случайные 6 000 рядов')
axes[0].axvline(1.32, color='gray', linestyle='--')
axes[0].axhline(.49, color='gray', linestyle='--')
axes[0].legend(fontsize=7)
segments.set_index('segment')[['series_share', 'units_share']].plot.bar(ax=axes[1])
axes[1].set(title='Доля рядов и доля продаж', xlabel='', ylabel='Доля')
savefig(fig, '03_demand_segments')


fig, axes = plt.subplots(2, 2, figsize=(13, 6), sharex=True)
for ax, label in zip(axes.flat, ['smooth', 'erratic', 'intermittent', 'lumpy']):
    candidates = profile.index[profile.segment == label]
    if len(candidates):
        idx = (profile.loc[candidates, 'units_last_365'] - profile.loc[candidates, 'units_last_365'].median()).abs().idxmin()
        ax.plot(dates[-180:], train[idx, -180:], linewidth=.8)
        ax.set(title=f'{label}: {meta.id.iloc[idx].removesuffix("_validation")}', ylabel='Единиц')
savefig(fig, '04_representative_series')
note('Один и тот же процент нулей может сопровождаться стабильным размером покупки '
     'или редкими крупными всплесками. Для разных сегментов нужны отдельные диагностики ошибки. '
     'В самом ADI нет информации об остатках, поэтому это сегментация наблюдаемых продаж.')

## 7. Недельная сезонность и эффект агрегации

Гипотеза: недельный цикл заметен в агрегате, но гораздо слабее на уровне отдельного товара.
Сравним корреляцию на лаге 7 и устойчивую STL-декомпозицию агрегированного ряда (`period=7`).
Силы тренда/сезонности как описательные величины; STL использует обе стороны окна,
поэтому его компоненты нельзя без изменений переносить в признаки прогнозной модели.

In [ ]:
decomposition = STL(np.log1p(total), period=7, robust=True).fit()
seasonal_strength = max(0., 1 - np.var(decomposition.resid) /
                        np.var(decomposition.seasonal + decomposition.resid))
trend_strength = max(0., 1 - np.var(decomposition.resid) /
                     np.var(decomposition.trend + decomposition.resid))

def row_corr(a, b):
    a, b = a.astype(np.float64), b.astype(np.float64)
    a -= a.mean(axis=1, keepdims=True)
    b -= b.mean(axis=1, keepdims=True)
    den = np.sqrt(np.square(a).sum(axis=1) * np.square(b).sum(axis=1))
    return np.divide((a * b).sum(axis=1), den, out=np.full(len(a), np.nan), where=den > 0)

lag7 = np.empty(N)
for lo in range(0, N, 1024):
    lag7[lo:lo + 1024] = row_corr(train[lo:lo + 1024, -365:-7], train[lo:lo + 1024, -358:])
aggregate_lag7 = float(np.corrcoef(total[-365:-7], total[-358:])[0, 1])
profile['lag7_corr_last365'] = lag7
seasonality_table = table(pd.DataFrame([
    {'level': 'Товар × магазин: медиана', 'lag7_correlation': np.nanmedian(lag7)},
    {'level': 'Все магазины: сумма', 'lag7_correlation': aggregate_lag7},
]), 'aggregation_seasonality')
table(pd.DataFrame([{'weekly_strength_log_total': seasonal_strength, 'trend_strength_log_total': trend_strength}]), 'stl_strength')

# Нормируем каждый магазин и каждую неделю: сравнение дней недели меньше зависит от тренда.
weekly_rows = []
full_weeks = eda_cal.groupby('wm_yr_wk').size().loc[lambda s: s == 7].index
for store, values in by_store.items():
    frame = eda_cal[['date', 'wm_yr_wk']].copy()
    frame['sales'] = values
    frame = frame[frame.wm_yr_wk.isin(full_weeks)]
    frame['ratio'] = frame.sales / frame.groupby('wm_yr_wk').sales.transform('mean')
    frame['dow'] = frame.date.dt.dayofweek
    weekly_rows.append(frame.groupby('dow').ratio.mean().rename(store))
weekday_profile = pd.DataFrame(weekly_rows)
weekday_profile.columns = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
table(weekday_profile.rename_axis('store').reset_index(), 'weekday_normalized')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(dates, np.log1p(total), alpha=.35, linewidth=.6, label='log(1 + продажи)')
axes[0].plot(dates, decomposition.trend, linewidth=1, label='STL тренд')
axes[0].legend(fontsize=8)
axes[0].set(title='Тренд агрегата и праздничные провалы')
sns.heatmap(weekday_profile, annot=True, fmt='.2f', center=1, cmap='vlag', ax=axes[1], cbar=False)
axes[1].set(title='День / среднее своей недели', xlabel='')
axes[2].hist(lag7[np.isfinite(lag7)], bins=60, density=True, alpha=.7)
axes[2].axvline(aggregate_lag7, color='red', label=f'Агрегат: {aggregate_lag7:.2f}')
axes[2].set(title='Корреляция на лаге 7: отдельные ряды', xlabel='Корреляция', ylabel='Плотность')
axes[2].legend(fontsize=8)
savefig(fig, '05_seasonality_and_aggregation')
note(f'Вывод: Корреляция на лаге 7: {aggregate_lag7:.2f} для агрегата и '
     f'{np.nanmedian(lag7):.2f} для медианного товарного ряда. Сила недельной сезонности '
     f'log-агрегата по STL: {seasonal_strength:.2f}. Недельные признаки разумны, '
     'но качество агрегата не характеризует качество каждого товара.')

## 8. Рост сети или расширение наблюдаемого ассортимента?

Сравним средние за первые и последние 364 дня EDA:
`продажи в день = число рядов с ценой × продажи на такой ряд`
помогает отделить изменение экспозиции от изменения интенсивности. Дополнительно возьмём
фиксированную когорту товаров с ценой на протяжении обоих окон

In [ ]:
early_idx = np.arange(364)
late_idx = np.arange(CUT - 364, CUT)
cohort = np.isfinite(P[:, np.unique(week_index[early_idx])]).all(axis=1)
cohort &= np.isfinite(P[:, np.unique(week_index[late_idx])]).all(axis=1)
growth_rows = []
for name, idx in [('Первые 364 дня', early_idx), ('Последние 364 дня', late_idx)]:
    day_units = float(total[idx].mean())
    exposure = float(active_count[idx].mean())
    growth_rows.append({'period': name, 'daily_units': day_units, 'mean_series_with_price': exposure,
                        'units_per_exposed_series_day': day_units / exposure,
                        'fixed_cohort_daily_units': float(train[cohort][:, idx].sum(dtype=np.int64) / len(idx))})
growth = table(pd.DataFrame(growth_rows), 'assortment_growth')
growth_total = growth.daily_units.iloc[1] / growth.daily_units.iloc[0] - 1
growth_exposure = growth.mean_series_with_price.iloc[1] / growth.mean_series_with_price.iloc[0] - 1
growth_intensity = growth.units_per_exposed_series_day.iloc[1] / growth.units_per_exposed_series_day.iloc[0] - 1
cohort_growth = growth.fixed_cohort_daily_units.iloc[1] / growth.fixed_cohort_daily_units.iloc[0] - 1
assert np.isclose(1 + growth_total, (1 + growth_exposure) * (1 + growth_intensity))
note(f'Продажи в день изменились на {growth_total:+.1%}; число рядов с ценой - '
     f'на {growth_exposure:+.1%}; продажи на экспонированный ряд - на {growth_intensity:+.1%}. '
     f'У фиксированной когорты из {cohort.sum():,} рядов изменение {cohort_growth:+.1%}. '
     'Состав ассортимента влияет на общий тренд; сравнение когорт всё ещё подвержено survivor bias '
     'и различиям календарных окон.')

## 9. SNAP: сырое сравнение и сравнение с календарными контролями

Гипотеза: связь SNAP с продажами различается по штатам и категориям, где соответственно
SNAP - это программа продовольственной помощи США; используем её календарные индикаторы из M5.
Сначала сравним средние, затем оценим отдельную OLS-модель для каждого штат x категория:
`log(1 + sales) ~ SNAP + day_of_week + year_month + event_type`.
Используем последние 3 × 365 дней EDA и HAC-интервалы с лагом 28 для временной зависимости.

Интерпретируем коэффициент как условную ассоциацию а не как "причинный" эффект:
SNAP связан с днём месяца, доходом и составом покупателей. Чувствительность к контролю дня
месяца проверим отдельно

In [ ]:
snap_rows, snap_models = [], {}
window = min(3 * 365, CUT)
for (state, cat), ix in meta.groupby(['state_id', 'cat_id'], observed=True).indices.items():
    frame = eda_cal.iloc[-window:].copy()
    frame['sales'] = train[ix, -window:].sum(axis=0, dtype=np.int64)
    frame['log_sales'] = np.log1p(frame.sales)
    frame['snap'] = frame[f'snap_{state}']
    frame['dow'] = frame.date.dt.dayofweek
    frame['ym'] = frame.date.dt.to_period('M').astype(str)
    frame['dom'] = frame.date.dt.day
    frame['event'] = frame.event_type_1.fillna('None')
    formula = 'log_sales ~ snap + C(dow) + C(ym) + C(event)'
    model = smf.ols(formula, data=frame).fit(cov_type='HAC', cov_kwds={'maxlags': 28})
    lo, hi = model.conf_int().loc['snap']
    # Диагностика доступной вариации SNAP после контроля дня месяца.
    snap_residual = smf.ols('snap ~ C(dom) + C(ym)', data=frame).fit().resid
    residual_variation = float(np.std(snap_residual))
    sensitivity = smf.ols(formula + ' + C(dom)', data=frame).fit(cov_type='HAC', cov_kwds={'maxlags': 28})
    identifiable = residual_variation > 1e-6
    snap_rows.append({
        'state': str(state), 'category': str(cat),
        'raw_pct': 100 * (frame.loc[frame.snap == 1, 'sales'].mean() / frame.loc[frame.snap == 0, 'sales'].mean() - 1),
        'adjusted_pct': 100 * np.expm1(model.params['snap']),
        'ci_low_pct': 100 * np.expm1(lo), 'ci_high_pct': 100 * np.expm1(hi),
        'dom_adjusted_pct': 100 * np.expm1(sensitivity.params['snap']) if identifiable else np.nan,
        'snap_residual_std_after_dom': residual_variation, 'days': len(frame),
    })
    snap_models[(str(state), str(cat))] = model
snap_results = table(pd.DataFrame(snap_rows), 'snap_association')
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(snap_results))
ax.scatter(snap_results.raw_pct, x + .12, label='Сырое сравнение', marker='x')
ax.errorbar(snap_results.adjusted_pct, x - .12,
            xerr=[snap_results.adjusted_pct - snap_results.ci_low_pct,
                  snap_results.ci_high_pct - snap_results.adjusted_pct], fmt='o', capsize=3,
            label='С календарными контролями, 95% HAC CI')
ax.set_yticks(x, snap_results.state + ' / ' + snap_results.category)
ax.axvline(0, color='gray', linestyle='--')
ax.set(title='SNAP: ассоциация, а не оценка причинного эффекта', xlabel='Изменение, %')
ax.legend(fontsize=8)
savefig(fig, '06_snap_adjustment')
note('Интервалы описывают неопределённость условной регрессии, не устраняют '
     'скрытые факторы и не скорректированы за девять сравнений. `NaN` в `dom_adjusted_pct` означает, '
     'что после контроля дня месяца независимой вариации SNAP практически нет; такой коэффициент не интерпретируем.')

## 10. Праздники и аномалии

Возьмём самые сильные отрицательные отклонения от STL-тренда и недельного цикла.
Дополнительно сравним Рождество с тем же днём недели за 7 и 14 дней.
Будущие дни в таком сравнении допустимы для EDA,
но недопустимы как признаки онлайн-прогноза. Выбросы соответственно пока не удаляем

In [ ]:
residual = decomposition.resid
mad = np.median(np.abs(residual - np.median(residual)))
robust_z = (residual - np.median(residual)) / max(1.4826 * mad, 1e-9)
anomalies = eda_cal[['date', 'event_name_1', 'event_name_2']].copy()
anomalies['sales'] = total
anomalies['robust_z_log'] = robust_z
table(anomalies.nsmallest(12, 'robust_z_log'), 'negative_anomalies')
holiday_rows = []
christmas = np.flatnonzero((eda_cal.event_name_1 == 'Christmas') | (eda_cal.event_name_2 == 'Christmas'))
for idx in christmas:
    neighbors = np.array([idx - 14, idx - 7, idx + 7, idx + 14])
    neighbors = neighbors[(neighbors >= 0) & (neighbors < CUT)]
    reference = np.median(total[neighbors])
    holiday_rows.append({'date': dates[idx], 'sales': total[idx], 'same_weekday_reference': reference,
                         'ratio_to_reference': total[idx] / reference, 'controls': len(neighbors)})
holiday_table = table(pd.DataFrame(holiday_rows), 'christmas_diagnostic')
fig, ax = plt.subplots(figsize=(11, 4))
for idx in christmas:
    left, right = max(0, idx - 21), min(CUT, idx + 22)
    reference = np.median(total[max(0, idx - 28):idx])
    ax.plot(np.arange(left, right) - idx, total[left:right] / reference, label=str(dates[idx].year), alpha=.8)
ax.axvline(0, color='gray', linestyle='--')
ax.set(title='Продажи вокруг Рождества', xlabel='Дней относительно 25 декабря', ylabel='К медиане предыдущих 28 дней')
ax.legend(ncol=5)
savefig(fig, '07_holiday_diagnostic')
note('Календарно повторяющийся провал получился полезным сигналом, '
     'а не достаточное основание для winsorization или заполнения средним. Причину закрытия магазина '
     'из этих файлов установить нельзя. При мониторинге такие дни нужно сравнивать с календарным ожиданием.')

## 11. Изменения цены: сравнение внутри ряда с контрольной группой

Гипотеза: падение цены сопровождается изменением продаж, но корреляция между
дорогими и дешёвыми товарами смешивает разные товары и объёмы спроса

Поэтому используем последние 104 полные недели EDA. Событие - это изменение цены минимум на 5%;
сравнение - изменение `log(1 + weekly_sales)` относительно предыдущей недели. Из него вычитаем
медианное изменение товаров с ценой 1% в том же магазин x категория x неделя
(минимум 20 контролей). 1 000 bootstrap-перевыборок целых товарных рядов дают 95% интервалы
Это диагностическое сравнение "до/после": без информации о промо и запасах оно не оценивает эластичность

In [ ]:
full_positions = np.flatnonzero(week_day_counts == 7)[-104:]
pw = P[:, eda_week_ids[full_positions]]
sw = W[:, full_positions]
with np.errstate(divide='ignore', invalid='ignore'):
    price_change = pw[:, 1:] / pw[:, :-1] - 1
sales_change = np.diff(np.log1p(sw.astype(np.float32)), axis=1)
valid = np.isfinite(price_change)
controls = valid & (np.abs(price_change) <= .01)
control_change = np.full_like(sales_change, np.nan)
for _, ix in meta.groupby(['store_id', 'cat_id'], observed=True).indices.items():
    for w in range(sales_change.shape[1]):
        selected = ix[controls[ix, w]]
        if len(selected) >= 20:
            control_change[ix, w] = np.median(sales_change[selected, w])
events = []
for label, mask in [('Снижение ≥5%', price_change <= -.05), ('Повышение ≥5%', price_change >= .05)]:
    rr, cc = np.where(valid & mask & np.isfinite(control_change))
    event_day = np.array([np.flatnonzero(week_index[:CUT] == w)[0] for w in eda_week_ids[full_positions]])[cc + 1]
    events.append(pd.DataFrame({'series': rr, 'week_position': cc, 'direction': label,
                                'price_pct': price_change[rr, cc] * 100,
                                'previous_units': sw[rr, cc], 'age_at_event': event_day - first_sale[rr],
                                'raw_log_change': sales_change[rr, cc],
                                'adjusted_log_change': sales_change[rr, cc] - control_change[rr, cc]}))
price_events = pd.concat(events, ignore_index=True)
table(price_events, 'price_change_events', show=False)
price_rows = []
for label, frame in price_events.groupby('direction'):
    cluster = frame.groupby('series').adjusted_log_change.agg(['sum', 'count'])
    values, counts = cluster['sum'].to_numpy(), cluster['count'].to_numpy()
    boot_rng = np.random.default_rng(SEED)
    boot = np.empty(1000)
    for b in range(len(boot)):
        ix = boot_rng.integers(0, len(cluster), len(cluster))
        boot[b] = values[ix].sum() / counts[ix].sum()
    lower, upper = np.quantile(boot, [.025, .975])
    established = frame[(frame.previous_units > 0) & (frame.age_at_event >= 90)]
    price_rows.append({'direction': label, 'events': len(frame), 'series': len(cluster),
                       'raw_log_change': frame.raw_log_change.mean(),
                       'adjusted_log_change': frame.adjusted_log_change.mean(),
                       'ci_low': lower, 'ci_high': upper,
                       'previous_zero_share': frame.previous_units.eq(0).mean(),
                       'established_events': len(established),
                       'established_adjusted_log_change': established.adjusted_log_change.mean()})
price_results = table(pd.DataFrame(price_rows), 'price_association')
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.errorbar(price_results.adjusted_log_change, np.arange(len(price_results)),
            xerr=[price_results.adjusted_log_change - price_results.ci_low,
                  price_results.ci_high - price_results.adjusted_log_change], fmt='o', capsize=5)
ax.set_yticks(np.arange(len(price_results)), price_results.direction)
ax.axvline(0, color='gray', linestyle='--')
ax.set(title='Изменение продаж сверх контроля: 95% cluster-bootstrap CI',
       xlabel='Разность изменений log(1 + недельные продажи)')
savefig(fig, '08_price_association')
note(f'Проверено {len(price_events):,} событий. Контрольная группа частично учитывает общий календарный '
     'шок магазина и категории. Отбор товаров в промо, замещение и обратная причинность остаются; '
     'bootstrap по товарам также не устраняет общие шоки между товарами. Цена на будущие недели '
     'может быть признаком только если её план действительно доступен в момент прогноза.')
for row in price_results.itertuples():
    note(f'Для {row.direction} оценка меняется с {row.adjusted_log_change:+.3f} '
         f'до {row.established_adjusted_log_change:+.3f} log-единиц; '
         f'у {row.previous_zero_share:.1%} исходных событий предыдущая неделя нулевая. '
         'Эта чувствительность не позволяет интерпретировать исходную оценку как эластичность.')
del pw, sw, price_change, sales_change, control_change, controls, valid
gc.collect()

## 12. Проверка EDA прогнозом: пять простых ориентиров

Без подбора параметров сравним `zero` (диагностика разреженности), `seasonal_naive_7`
(повтор последней недели), `mean_28` и `mean_56` (средние последних 28 и 56 дней), `weekday_mean_56`
(среднее по соответствующему дню недели за 8 недель). Каждый прогноз использует
только историю до начала своего 28-дневного окна

Основная диагностическая метрика:
\(WAPE=\sum|y-\hat y|/\sum y\). Она определена при нулях отдельных товаров, но отдаёт больший вес
массовым продажам. Дополнительно: MAE, signed bias (положительный — перепрогноз), WAPE агрегата,
среднее RMSSE товарных рядов и RMSSE с весами по выручке последних 28 дней истории.

\(RMSSE_i=\sqrt{MSE_{i,test}/mean((y_{i,t}-y_{i,t-1})^2)}\), знаменатель берётся из истории
после первой ненулевой продажи. Ряды с нулевым знаменателем исключаются из RMSSE,
их число выводится; для весов оставшиеся веса нормируются заново.
`revenue_weighted_RMSSE_bottom` относится только к нижнему уровню;
это не официальный WRMSSE M5 по 12 уровням иерархии. MAPE не используем из-за нулей.

In [ ]:
def forecasts(history):
    n, t = history.shape
    future_dow = np.arange(t, t + HORIZON) % 7
    past_dow = np.arange(t - 56, t) % 7
    weekday_means = np.column_stack([history[:, -56:][:, past_dow == d].mean(axis=1) for d in range(7)])
    return {
        'zero': np.zeros((n, HORIZON), dtype=np.float32),
        'seasonal_naive_7': np.tile(history[:, -7:], (1, 4)).astype(np.float32),
        'mean_28': np.repeat(history[:, -28:].mean(axis=1)[:, None], HORIZON, axis=1).astype(np.float32),
        'mean_56': np.repeat(history[:, -56:].mean(axis=1)[:, None], HORIZON, axis=1).astype(np.float32),
        'weekday_mean_56': weekday_means[:, future_dow].astype(np.float32),
    }

def rmsse_scales(history):
    n, t = history.shape
    scale = np.full(n, np.nan)
    for lo in range(0, n, 1024):
        block = history[lo:lo + 1024]
        positive = block > 0
        first = np.where(positive.any(axis=1), positive.argmax(axis=1), t)
        diff = np.diff(block.astype(np.float32), axis=1)
        valid_diff = np.arange(1, t)[None, :] > first[:, None]
        count = valid_diff.sum(axis=1)
        sums = (np.square(diff) * valid_diff).sum(axis=1, dtype=np.float64)
        scale[lo:lo + len(block)] = np.divide(sums, count, out=np.full(len(block), np.nan), where=count > 0)
    return scale

backtest_rows, segment_rows, category_rows, daily_predictions = [], [], [], []
item_error_sums = {name: np.zeros(N) for name in ['zero', 'seasonal_naive_7', 'mean_28', 'mean_56', 'weekday_mean_56']}
fold_definitions = []
for fold, origin in enumerate(range(CUT, T, HORIZON), 1):
    history = Y[:, :origin]
    actual = Y[:, origin:origin + HORIZON].astype(np.float32)
    scale = rmsse_scales(history)
    valid_scale = np.isfinite(scale) & (scale > 0)
    revenue = np.zeros(N, dtype=np.float64)
    for d in range(origin - 28, origin):
        day_price = P[:, week_index[d]]
        assert not np.any((history[:, d] > 0) & ~np.isfinite(day_price))
        revenue += history[:, d] * np.nan_to_num(day_price, nan=0.)
    weights = revenue[valid_scale] / revenue[valid_scale].sum()
    fold_definitions.append({'fold': fold, 'train_end': str(cal.date.iloc[origin - 1].date()),
                             'test_start': str(cal.date.iloc[origin].date()),
                             'test_end': str(cal.date.iloc[origin + HORIZON - 1].date()),
                             'rmsse_excluded': int((~valid_scale).sum())})
    for name, pred in forecasts(history).items():
        assert pred.shape == actual.shape and np.isfinite(pred).all() and (pred >= 0).all()
        err = pred - actual
        abs_by_series = np.abs(err).sum(axis=1, dtype=np.float64)
        rmsse = np.sqrt(np.mean(err[valid_scale]**2, axis=1) / scale[valid_scale])
        item_error_sums[name] += abs_by_series
        backtest_rows.append({'fold': fold, 'model': name,
            'WAPE': abs_by_series.sum() / actual.sum(dtype=np.float64),
            'MAE': np.abs(err).mean(dtype=np.float64), 'mean_RMSSE': rmsse.mean(),
            'revenue_weighted_RMSSE_bottom': np.sum(weights * rmsse),
            'bias': err.sum(dtype=np.float64) / actual.sum(dtype=np.float64),
            'aggregate_WAPE': np.abs(err.sum(axis=0, dtype=np.float64)).sum() / actual.sum(dtype=np.float64),
            'rmsse_excluded': int((~valid_scale).sum())})
        for label in np.unique(segment):
            ix = segment == label
            denominator = actual[ix].sum(dtype=np.float64)
            segment_rows.append({'fold': fold, 'model': name, 'segment': label, 'series': int(ix.sum()),
                                 'absolute_error': float(abs_by_series[ix].sum()), 'actual_units': denominator,
                                 'WAPE': abs_by_series[ix].sum() / denominator if denominator > 0 else np.nan})
        for cat, ix in meta.groupby('cat_id', observed=True).indices.items():
            category_rows.append({'fold': fold, 'model': name, 'category': str(cat),
                                  'absolute_error': float(abs_by_series[ix].sum()),
                                  'actual_units': float(actual[ix].sum(dtype=np.float64))})
        daily_predictions.append(pd.DataFrame({'date': cal.date.iloc[origin:origin + HORIZON].to_numpy(),
                                              'fold': fold, 'model': name, 'actual': actual.sum(axis=0, dtype=np.float64),
                                              'prediction': pred.sum(axis=0, dtype=np.float64)}))
backtest = table(pd.DataFrame(backtest_rows), 'backtest_metrics')
folds = table(pd.DataFrame(fold_definitions), 'backtest_folds')
segment_metrics = table(pd.DataFrame(segment_rows), 'backtest_segments', show=False)
category_metrics = table(pd.DataFrame(category_rows), 'backtest_categories', show=False)
predictions_daily = table(pd.concat(daily_predictions, ignore_index=True), 'backtest_daily_predictions', show=False)
scoreboard = table(backtest.groupby('model', sort=False).agg(
    WAPE_mean=('WAPE', 'mean'), WAPE_std=('WAPE', 'std'), MAE=('MAE', 'mean'),
    mean_RMSSE=('mean_RMSSE', 'mean'), revenue_weighted_RMSSE_bottom=('revenue_weighted_RMSSE_bottom', 'mean'),
    bias=('bias', 'mean'), aggregate_WAPE=('aggregate_WAPE', 'mean')).reset_index().sort_values('WAPE_mean'), 'backtest_summary')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=backtest, x='fold', y='WAPE', hue='model', marker='o', ax=axes[0])
axes[0].set(title='WAPE по трём будущим окнам', xticks=[1, 2, 3], xlabel='28-дневное окно')
pooled = segment_metrics.groupby(['segment', 'model'])[['absolute_error', 'actual_units']].sum()
segment_wape = (pooled.absolute_error / pooled.actual_units).unstack()
sns.heatmap(segment_wape, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
axes[1].set(title='WAPE по сегментам, три окна вместе', xlabel='', ylabel='')
savefig(fig, '09_backtest_and_segments')
best_name = scoreboard.iloc[0].model
baseline_wape = scoreboard.set_index('model').loc['seasonal_naive_7', 'WAPE_mean']
best_wape = float(scoreboard.iloc[0].WAPE_mean)
note(f'Среди заданных ориентиров минимальный средний WAPE у `{best_name}`: '
     f'{best_wape:.3f}; относительно повторения последней недели изменение '
     f'{(best_wape / baseline_wape - 1):+.1%}. Это сравнение пяти фиксированных правил, '
     'а не доказательство превосходства над обучаемыми моделями. Три окна в феврале–апреле '
     'не покрывают годовую сезонность; отдельного финального теста после выбора победителя здесь нет.')

## 13. Проверка практической значимости и ошибки агрегирования

Сравним две гипотезы: `weekday_mean_56` против `mean_56`.
Одинаковая история из 56 дней изолирует эффект разделения по дням недели;
оба правила и сравнение фиксируются до просмотра результатов этого контроля.
Парный bootstrap по 3 049 товарам переносит вместе все 10 магазинов одного товара
и все три тестовых окна. Он оценивает неопределённость по ассортименту при фиксированных датах;
не оценивает неопределённость будущих календарных периодов.

In [ ]:
holdout_units = Y[:, CUT:].sum(axis=1, dtype=np.int64)
comparison = pd.DataFrame({'item_id': meta.item_id,
    'error_difference': item_error_sums['weekday_mean_56'] - item_error_sums['mean_56'],
    'actual_units': holdout_units}).groupby('item_id', observed=True)[['error_difference', 'actual_units']].sum()
arr = comparison.to_numpy()
boot_rng = np.random.default_rng(SEED)
deltas = np.empty(2000)
for b in range(len(deltas)):
    draw = arr[boot_rng.integers(0, len(arr), len(arr))]
    deltas[b] = draw[:, 0].sum() / draw[:, 1].sum()
delta = float(arr[:, 0].sum() / arr[:, 1].sum())
delta_lo, delta_hi = np.quantile(deltas, [.025, .975])
table(pd.DataFrame([{'comparison': 'weekday_mean_56 minus mean_56', 'delta_WAPE': delta,
                     'ci_low': delta_lo, 'ci_high': delta_hi, 'clusters_items': len(arr), 'bootstrap_samples': len(deltas)}]),
      'baseline_paired_bootstrap')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(deltas, bins=45, alpha=.8)
axes[0].axvline(0, color='gray', linestyle='--')
axes[0].set(title='Парный bootstrap по товарам', xlabel='ΔWAPE: weekday_mean_56 − mean_56', ylabel='Частота')
for name in ['mean_56', 'weekday_mean_56']:
    values = predictions_daily[predictions_daily.model == name]
    axes[1].plot(values.date, values.prediction, label=name)
axes[1].plot(values.date, values.actual, color='black', alpha=.6, label='Факт')
axes[1].set(title='Агрегированный прогноз на holdout', ylabel='Единиц в день')
axes[1].tick_params(axis='x', rotation=25)
axes[1].legend(fontsize=8)
savefig(fig, '10_bootstrap_and_aggregate_forecast')
sign_conclusion = ('интервал целиком ниже нуля' if delta_hi < 0 else
                   'интервал целиком выше нуля' if delta_lo > 0 else 'интервал пересекает ноль')
note(f'delta WAPE = {delta:+.4f}, 95% CI [{delta_lo:+.4f}; {delta_hi:+.4f}]; {sign_conclusion}. '
     'Длина истории одинакова; разница относится к разбиению по дням недели. Маленький WAPE '
     'суммарного ряда может скрывать взаимное погашение ошибок товаров. Контроль добавлен '
     'после первого исследовательского сравнения разных окон; результат остаётся exploratory, '
     'а не независимой подтверждающей проверкой.')
score_by_name = scoreboard.set_index('model')
agg_mean = score_by_name.loc['mean_56', 'aggregate_WAPE']
agg_weekday = score_by_name.loc['weekday_mean_56', 'aggregate_WAPE']
direction_text = 'увеличивает' if delta > 0 else 'уменьшает'
note(f'Разделение 56 дней по дням недели {direction_text} товарный WAPE '
     f'на {abs(delta) * 100:.2f} процентного пункта, тогда как WAPE агрегата меняется '
     f'с {agg_mean:.3f} до {agg_weekday:.3f}. Объединение редких продаж сглаживает шум; '
     'выбирать модель только по красивому графику общего объёма нельзя.')

## 14. Сдвиг распределения и кандидаты для мониторинга

Сравним последние 84 дня EDA с 84 днями backtest: объём, нули, состав продаж и изменение
по категориям. Это аудит после получения прогнозов. Большой объём данных делает обычные
тесты значимости слишком чувствительными, поэтому используем величины изменений.
Пороги алертов из одного такого сравнения не назначаем: нужны исторические сезонные backtest.

In [ ]:
reference, current = Y[:, CUT - 84:CUT], Y[:, CUT:]
drift_rows = []
for cat, ix in meta.groupby('cat_id', observed=True).indices.items():
    a, b = reference[ix], current[ix]
    drift_rows.append({'category': str(cat), 'reference_units': int(a.sum(dtype=np.int64)),
                       'current_units': int(b.sum(dtype=np.int64)),
                       'volume_change_pct': 100 * (b.sum(dtype=np.float64) / a.sum(dtype=np.float64) - 1),
                       'zero_share_reference': float((a == 0).mean()), 'zero_share_current': float((b == 0).mean()),
                       'zero_change_pp': 100 * ((b == 0).mean() - (a == 0).mean())})
drift = table(pd.DataFrame(drift_rows), 'drift_by_category')
category_ref = drift.reference_units.to_numpy() / drift.reference_units.sum()
category_cur = drift.current_units.to_numpy() / drift.current_units.sum()
mid = (category_ref + category_cur) / 2
js_divergence = float((stats.entropy(category_ref, mid, base=2) + stats.entropy(category_cur, mid, base=2)) / 2)
recent_reference_units = reference.sum(axis=1, dtype=np.int64)
newly_observed = (first_sale >= CUT) & (holdout_units > 0)
table(pd.DataFrame([{'category_JS_divergence_bits': js_divergence,
                     'series_first_positive_in_holdout': int(newly_observed.sum()),
                     'overall_volume_change_pct': 100 * (current.sum(dtype=np.float64) / reference.sum(dtype=np.float64) - 1)}]),
      'drift_summary')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=drift, x='category', y='volume_change_pct', ax=axes[0])
axes[0].axhline(0, color='gray')
axes[0].set(title='Изменение объёма: holdout к предыдущим 84 дням', xlabel='', ylabel='%')
sns.barplot(data=drift, x='category', y='zero_change_pp', ax=axes[1])
axes[1].axhline(0, color='gray')
axes[1].set(title='Изменение доли нулей', xlabel='', ylabel='Процентные пункты')
savefig(fig, '11_distribution_shift')
note(f'**Интерпретация.** JS-дивергенция состава продаж по категориям: {js_divergence:.5f} бит '
     '(0 — совпадающие доли). Это только три категории: стабильность этой величины не гарантирует '
     'стабильности отдельных товаров. Различия зимы и весны, праздники и изменение ассортимента '
     'нельзя автоматически объявлять поломкой данных или модели.')

## 15. Выводы и решения для следующего этапа

In [ ]:
best = scoreboard.iloc[0]
snap_food = snap_results[snap_results.category == 'FOODS']
snap_text = '; '.join(f'{r.state}: {r.adjusted_pct:+.1f}%' for r in snap_food.itertuples())
note(f'''
1. Качество и масштаб: проанализированы {N:,} рядов; исходная история - {T} дней,
   EDA - {CUT} дней. Все {len(quality)} проверок контракта пройдены; исходные файлы закреплены SHA-256.
2. Нули не равны отсутствию спроса: доля нулей {all_zero_share:.1%}; при известной цене -
   {known_zero_share:.1%}. Пропуск цены не заменяем нулём и не удаляем нулевые продажи.
3. Состав имеет значение: верхние 20% рядов дают {top20_share:.1%} единиц последнего года.
   Изменение объёма {growth_total:+.1%} между годовыми окнами сочетается с изменением наблюдаемой
   экспозиции {growth_exposure:+.1%}; нужны возраст товара и признаки доступности.
4. Агрегат скрывает трудность прогноза: лаг-7 корреляция агрегата {aggregate_lag7:.2f},
   медианная по товарам {np.nanmedian(lag7):.2f}. Проверяем ошибки по сегментам и уровням иерархии.
5. Календарь полезен, причинность не доказана: SNAP-ассоциации для FOODS после базовых
   календарных контролей: {snap_text}. Проверка дня месяца ограничивает интерпретацию.
   Праздничные провалы сохраняем как сигнал; ценовые события требуют учёта промо и запасов.
6. Проверка будущим: лучший из фиксированных ориентиров - `{best_name}`: WAPE {best.WAPE_mean:.3f},
   средний RMSSE {best.mean_RMSSE:.3f}; три временных окна. Для `weekday_mean_56 − mean_56`
   delta WAPE {delta:+.4f}, bootstrap CI [{delta_lo:+.4f}; {delta_hi:+.4f}].
   Длина истории в этом контроле одинакова: недельное разбиение {direction_text} товарный WAPE,
   а WAPE агрегата меняется с {agg_mean:.3f} до {agg_weekday:.3f}. Выбор лучшего правила требует нового теста.
''')
decisions = pd.DataFrame([
    ['Нули и ассортимент', 'Не импутировать продажи; missing price — отдельный флаг; возраст товара считать на cutoff',
     'Доля рядов без цены, новые товары, продажи без цены'],
    ['Сезонность', 'Признаки дня недели, календаря и лаги; rolling только из доступной истории',
     'WAPE и bias по магазину, категории, сегменту'],
    ['SNAP / цена', 'Использовать только известный на момент прогноза календарь и план цен; проверить ablation',
     'Пропуски признаков и изменения цены, отдельно по штатам'],
    ['Праздники', 'Сохранить аномалии с объяснимым календарным контекстом', 'Отклонение от сезонного ожидания'],
    ['Валидация', 'Расширить rolling-origin на полный год; будущий финальный тест; официальный WRMSSE по 12 уровням',
     'Разрыв между ошибкой агрегата и нижнего уровня'],
], columns=['finding', 'next_action', 'monitor'])
table(decisions, 'modeling_and_monitoring_decisions')
summary_metrics = {
    'n_series': N, 'n_days_eda': CUT, 'n_days_holdout': T - CUT,
    'zero_share_all': all_zero_share, 'zero_share_known_price': known_zero_share,
    'missing_price_share': missing_price_share, 'positive_units_without_price': no_price_sales,
    'top20_units_share': top20_share, 'weekly_strength_log_total': seasonal_strength,
    'aggregate_lag7_correlation': aggregate_lag7, 'median_item_lag7_correlation': float(np.nanmedian(lag7)),
    'best_mean_WAPE': best_wape, 'weekday_vs_mean56_delta_WAPE': delta,
    'weekday_vs_mean56_ci_low': float(delta_lo), 'weekday_vs_mean56_ci_high': float(delta_hi),
    'category_JS_divergence_bits': js_divergence,
}
(OUT / 'summary_metrics.json').write_text(json.dumps(summary_metrics, indent=2), encoding='utf-8')
table(profile, 'series_profile', show=False)

analysis_config = {
    'seed': SEED, 'eda_end': str(eda_cal.date.max().date()), 'horizon_days': HORIZON,
    'n_folds': N_FOLDS, 'mirror_commit': manifest['mirror_commit'],
    'price_event_threshold': .05, 'snap_hac_lags': 28,
    'primary_diagnostic_metric': 'WAPE', 'official_M5_WRMSSE': False,
    'best_fixed_baseline': best_name,
}
(OUT / 'analysis_config.json').write_text(json.dumps(analysis_config, indent=2), encoding='utf-8')
# Позволяет загрузчику отличить завершённый анализ от частично обновлённых файлов.
artifact_hashes = {str(p.relative_to(OUT).as_posix()): file_hash(p)
                   for p in sorted(OUT.rglob('*')) if p.is_file() and p.name != 'analysis_manifest.json'}
(OUT / 'analysis_manifest.json').write_text(json.dumps({'files': artifact_hashes}, indent=2), encoding='utf-8')
print('EDA завершён. Артефакты готовы для scripts/log_eda.py.')